# airbench TTA on the 0.915 model (no training)

Runtime > Change runtime type > any GPU (L4 / T4 / A100), then Runtime > Run all. About 2-5 minutes on a GPU.

Same trained weights as the 0.915 submission; only the prediction changes: 6 views (image, mirror, 1-px shifts down-right and up-left, each with mirror), logits averaged with weights 0.5 (unshifted pair) / 0.25 / 0.25, as in airbench (Jordan 2024, *94% on CIFAR-10 in 3.29 seconds*). Still one network; nothing is learned from test images.

Decision: chosen on VALIDATION only. A new `submission.csv` is written only if airbench beats the current setting on val by more than 0.2 pt (5 of 2,500 images). The baseline row must show val 0.9852 / internal test 0.9812, which confirms the right model was loaded.


## 1. Paths, Drive, device

In [1]:
# ================= TTA test for ONE trained student (standalone: Colab or HPC) =================
# Loads best.pt of the 0.915 run, scores the current TTA (flip) and the airbench TTA on VAL and INTERNAL TEST,
# chooses on VAL only, and writes a Kaggle submission.csv only if airbench beats the current setting on VAL by > 0.2 pt.
# Single network, more views of the same image (same kind of TTA as the flip TTA already used).
import os, re, glob, time, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- paths: set RUN_NAME if you know it; otherwise the run whose submission has internal test 0.9812 is found automatically
RUN_NAME = None                       # e.g. "kd_sewrn_only_student_..." (the 0.915 run)
FIND_TEST_TAG = "test0.9812"          # used only when RUN_NAME is None
try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive; drive.mount("/content/drive")
    ROOT, WORK_DIR = "/content/drive/MyDrive/cifar10_resnet_challenge", "/content/work"
except ImportError:
    IN_COLAB = False
    ROOT = os.environ.get("TTA_ROOT", "/scratch/ds8223/cifar10_resnet_challenge"); WORK_DIR = os.environ.get("TTA_WORK", "/scratch/ds8223/kd_work")
CACHE_DIR, SPLIT_DIR = os.path.join(ROOT, "data", "cache"), os.path.join(ROOT, "data", "splits")

if RUN_NAME is None:
    hits = sorted({os.path.basename(os.path.dirname(p)) for p in glob.glob(os.path.join(ROOT, "runs", "*", f"submission_*{FIND_TEST_TAG}*.csv"))})
    print("runs with a submission tagged", FIND_TEST_TAG, ":", hits)
    assert len(hits) == 1, "found 0 or several runs: set RUN_NAME by hand (see the list above)"
    RUN_NAME = hits[0]
RUN_DIR = os.path.join(ROOT, "runs", RUN_NAME)
BEST_CKPT = os.path.join(RUN_DIR, "best.pt")
print("run:", RUN_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (slow: use a GPU)")
AMP_DTYPE = torch.bfloat16 if device.type == "cpu" or torch.cuda.is_bf16_supported() else None

Mounted at /content/drive
runs with a submission tagged test0.9812 : ['kd_sewrn_only_student_resnext29_slasd_400ep_v1']
run: kd_sewrn_only_student_resnext29_slasd_400ep_v1
device: Tesla T4


## 2. Model code (same classes as the training notebooks)

In [2]:
# ---- Models (same code as the training notebooks): ResNet, ResNeXt-29, Shake-Shake, chosen with CFG["ARCH"] ----
class BasicBlock(nn.Module):
    """Post-activation basic block (our original ResNet)."""
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class CifarResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256), blocks=(3, 3, 3), num_classes=10, dropout=0.0):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, cin = [], widths[0]
        for i, (w, nb) in enumerate(zip(widths, blocks)):
            for j in range(nb):
                layers.append(BasicBlock(cin, w, 2 if (j == 0 and i > 0) else 1)); cin = w
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn2.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class ResNeXtBlock(nn.Module):
    """ResNeXt bottleneck: 1x1 conv -> grouped 3x3 conv (cardinality groups) -> 1x1 conv, plus skip (Xie et al. 2017)."""
    def __init__(self, cin, inner, cout, stride, cardinality):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, inner, 1, bias=False); self.bn1 = nn.BatchNorm2d(inner)
        self.conv2 = nn.Conv2d(inner, inner, 3, stride, 1, groups=cardinality, bias=False); self.bn2 = nn.BatchNorm2d(inner)
        self.conv3 = nn.Conv2d(inner, cout, 1, bias=False); self.bn3 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.bn3(self.conv3(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class ResNeXt29(nn.Module):
    """CIFAR ResNeXt-29: 3x3 stem (64) + 3 stages x 3 blocks; output widths 256/512/1024;
    inner width = cardinality * bottleneck_width, doubled at each stage. (2, 64) is the standard 2x64d."""
    def __init__(self, cardinality=2, bottleneck_width=64, blocks=(3, 3, 3), out_widths=(256, 512, 1024),
                 num_classes=10, dropout=0.0, stem=64, strides=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, stem, 3, 1, 1, bias=False), nn.BatchNorm2d(stem), nn.ReLU(inplace=True))
        layers, cin = [], stem
        for i, (cout, nb) in enumerate(zip(out_widths, blocks)):
            inner = cardinality * bottleneck_width * (2 ** i)
            for j in range(nb):
                layers.append(ResNeXtBlock(cin, inner, cout, strides[i] if j == 0 else 1, cardinality)); cin = cout
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn3.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class SE(nn.Module):
    """Squeeze-excitation (Hu et al. 2018): channel weights from global average pooling. Teachers only."""
    def __init__(self, c, r=16):
        super().__init__()
        self.fc1, self.fc2 = nn.Conv2d(c, c // r, 1), nn.Conv2d(c // r, c, 1)
    def forward(self, x):
        return x * torch.sigmoid(self.fc2(F.relu(self.fc1(F.adaptive_avg_pool2d(x, 1)))))

class WBlock(nn.Module):
    """Pre-activation wide residual block (Zagoruyko & Komodakis 2016)."""
    def __init__(self, cin, cout, stride, drop, se_r=0):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(cin), nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(cout), nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.dropout = nn.Dropout(drop)
        self.skip = None if (stride == 1 and cin == cout) else nn.Conv2d(cin, cout, 1, stride, bias=False)
        if se_r: self.se = SE(cout, se_r)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.skip is None else self.skip(o)
        o = self.conv2(self.dropout(F.relu(self.bn2(self.conv1(o)))))
        if hasattr(self, "se"): o = self.se(o)
        return o + sc

class WRN(nn.Module):
    """WideResNet for CIFAR (Zagoruyko & Komodakis 2016). k = widen factor (WRN-28-10: 36.5M params) or explicit stage widths."""
    def __init__(self, depth=28, k=10, dropout=0.0, num_classes=10, se_r=0):
        super().__init__()
        n = (depth - 4) // 6
        w = [16] + (list(k) if isinstance(k, (tuple, list)) else [16 * k, 32 * k, 64 * k])   # k: widen factor, or the 3 stage widths
        self.stem = nn.Conv2d(3, w[0], 3, 1, 1, bias=False)
        layers, cin = [], w[0]
        for i in range(3):
            for j in range(n):
                layers.append(WBlock(cin, w[i + 1], 2 if (j == 0 and i > 0) else 1, dropout, se_r)); cin = w[i + 1]
        self.layers = nn.Sequential(*layers); self.bn = nn.BatchNorm2d(cin)
        self.drop = nn.Dropout(0.0); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(F.relu(self.bn(self.layers(self.stem(x)))), 1), 1)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))

def _build_base(c):
    """Build any of our architectures from a config dict (also rebuilds teachers from their saved cfg)."""
    arch = c.get("ARCH", "resnet")
    if arch == "resnet":
        return CifarResNet(c["WIDTHS"], c["BLOCKS"], dropout=c.get("DROPOUT", 0.0))
    if arch == "resnext":
        return ResNeXt29(c.get("CARDINALITY", 2), c.get("BOTTLENECK_WIDTH", 64), c.get("BLOCKS", (3, 3, 3)),
                         c.get("WIDTHS", (256, 512, 1024)), dropout=c.get("DROPOUT", 0.0),
                         strides=tuple(c.get("STRIDES", (1, 2, 2))))   # (1, 1, 2) = higher-resolution student, same params
    if arch in ("wrn", "sewrn"):     # sewrn = WRN with SE blocks: a training-only teacher
        return WRN(c["DEPTH"], c["WIDEN"], c.get("DROPOUT", 0.0), se_r=c.get("SE_REDUCTION", 16) if arch == "sewrn" else 0)
    raise ValueError(f"unknown ARCH {arch}")

def build_model(c):
    """Base network; with SLA on, adds the joint (class x rotation) head used only during training."""
    net = _build_base(c)
    if c.get("SLA", False):
        net.fc_joint = nn.Linear(net.fc.in_features, 10 * c.get("SLA_ROTATIONS", 4))
    return net

def forward_both(net, x):
    """Returns (10-class self-distillation head logits, joint class x rotation head logits)."""
    f = net.drop(net.features(x))
    return net.fc(f), net.fc_joint(f)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 3. Load best.pt of the 0.915 run, data and the same split

In [3]:
# ---- checkpoint, data, split (same split file as every run)
best = torch.load(BEST_CKPT, map_location=device, weights_only=False)
CFG = best["cfg"]
X_full = np.load(os.path.join(CACHE_DIR, "train_x.npy")); y_full = np.load(os.path.join(CACHE_DIR, "train_y.npy"))
X_test = np.load(os.path.join(CACHE_DIR, "test_x.npy")); test_ids = np.load(os.path.join(CACHE_DIR, "test_ids.npy"))
SAMPLE_SUB = pd.read_csv(os.path.join(CACHE_DIR, "sample_submission.csv"))
sp = np.load(os.path.join(SPLIT_DIR, f"split_seed{CFG['SPLIT_SEED']}_val{CFG['VAL_FRAC']}_test{CFG['TEST_FRAC']}_n{len(y_full)}.npz"))
va_idx, te_idx = sp["val"], sp["test"]
def to_dev(x): return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().to(device)
Xva, yva = to_dev(X_full[va_idx]), torch.from_numpy(y_full[va_idx]).to(device)
Xte_int, yte_int = to_dev(X_full[te_idx]), torch.from_numpy(y_full[te_idx]).to(device)
Xkaggle = to_dev(X_test)
MEAN, STD = best["norm"]["mean"].to(device), best["norm"]["std"].to(device)   # exact normalisation of that run
def normalize(x_u8): return ((x_u8.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def _ag_logits(m, xb):
    """SLA+AG: average the joint-head logit of (class, k) on the image rotated by k*90 degrees."""
    agg = 0
    for k in range(CFG.get("SLA_ROTATIONS", 4)):
        _, j = forward_both(m, torch.rot90(xb, k, dims=(2, 3)))
        agg = agg + j.float().view(len(xb), 10, -1)[:, :, k]
    return agg / CFG.get("SLA_ROTATIONS", 4)

net = build_model(CFG).to(device).to(memory_format=torch.channels_last)
net.load_state_dict(best["model"]); net.eval()
has_joint = getattr(net, "fc_joint", None) is not None
print(f"params {count_params(net):,} | saved val {best['val_acc']:.4f} | SLA head: {has_joint} | val {len(Xva)} | internal test {len(Xte_int)} | kaggle {len(Xkaggle)}")

params 4,998,738 | saved val 0.9856 | SLA head: True | val 2500 | internal test 2500 | kaggle 2000


## 4. airbench TTA vs current TTA: choose on val, check on internal test

In [4]:
# ---- airbench TTA (Jordan 2024, airbench code): 0.5 * mean(logits of image, mirror) + 0.5 * mean(logits of the
#      image shifted 1 px down-right and 1 px up-left, each with its mirror); reflect padding; LOGITS are averaged.
def shift(x, dy, dx, p=2):
    """Move the image content by (dy, dx) pixels; borders filled by reflection."""
    if dy == 0 and dx == 0: return x
    xp = F.pad(x, (p, p, p, p), mode="reflect")
    return xp[:, :, p - dy:p - dy + 32, p - dx:p - dx + 32].contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def view_logits(X, dy, dx, flip, bs=500):
    """Logits of the 10-class head (sd) and of the SLA aggregation (ag) for one view of every image."""
    sd, ag = [], []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        if flip: xb = xb.flip(3)
        xb = shift(xb, dy, dx)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            s = net(xb).float()
            a = _ag_logits(net, xb).float() if has_joint else s
        sd.append(s); ag.append(a)
    return torch.cat(sd), torch.cat(ag)

VIEWS = [(0, 0), (1, 1), (-1, -1)]                 # airbench: pad 1, crops [0:32] and [2:34]
DATA = {"val": (Xva, yva), "test": (Xte_int, yte_int), "kaggle": (Xkaggle, None)}
t0 = time.time()
cache = {name: {(dy, dx, fl): view_logits(X, dy, dx, fl) for (dy, dx) in VIEWS for fl in (False, True)}
         for name, (X, _) in DATA.items()}
print(f"computed {len(VIEWS) * 2} views x 3 sets in {time.time() - t0:.0f}s")

def head(v, mode):                                   # v = (sd_logits, ag_logits)
    return v[0] if mode == "sd" else v[1]

def scores(name, setting, mode):
    c = cache[name]
    def one(md):
        if setting == "flip only (current)":        # exactly how the 0.915 model was scored: sum of softmax over image + mirror
            return head(c[(0, 0, False)], md).softmax(1) + head(c[(0, 0, True)], md).softmax(1)
        mir = lambda dy, dx: 0.5 * (head(c[(dy, dx, False)], md) + head(c[(dy, dx, True)], md))
        return (0.5 * mir(0, 0) + 0.25 * mir(1, 1) + 0.25 * mir(-1, -1)).softmax(1)   # airbench weighting on logits
    return one(mode) if mode != "sdag" else 0.5 * (one("sd") / one("sd").sum(1, keepdim=True) + one("ag") / one("ag").sum(1, keepdim=True))

modes = ("sd", "ag", "sdag") if has_joint else ("sd",)
rows = []
for setting in ("flip only (current)", "airbench 6-view"):
    for md in modes:
        r = {"setting": setting, "mode": md}
        for name in ("val", "test"):
            r[name] = (scores(name, setting, md).argmax(1) == DATA[name][1]).float().mean().item()
        rows.append(r)
res = pd.DataFrame(rows)

# baseline = what the 0.915 submission used: flip, mode by the section-12 rule (sd unless ag/sdag wins VAL by > 0.2 pt)
b = res[res.setting == "flip only (current)"].set_index("mode")
alt = max([m for m in modes if m != "sd"], key=lambda m: b.loc[m, "val"], default="sd")
base_mode = alt if alt != "sd" and b.loc[alt, "val"] > b.loc["sd", "val"] + 0.002 else "sd"
base = res[(res.setting == "flip only (current)") & (res["mode"] == base_mode)].iloc[0]
air = res[res.setting == "airbench 6-view"].sort_values("val", ascending=False).iloc[0]
choice = air if air.val > base.val + 0.002 else base

print(res.assign(val=res.val.round(4), test=res.test.round(4)).to_string(index=False))
print(f"\nBASELINE (0.915 submission): flip only, mode '{base_mode}': val {base.val:.4f} | internal test {base.test:.4f}   (expected 0.9852 / 0.9812)")
print(f"BEST AIRBENCH ON VAL: mode '{air['mode']}': val {air.val:.4f} | internal test {air.test:.4f} "
      f"-> internal test change {100 * (air.test - base.test):+.2f} pt ({int(round((air.test - base.test) * 2500)):+d} images of 2,500)")

if choice.setting != base.setting:
    kp = scores("kaggle", choice.setting, choice["mode"]).argmax(1).cpu().numpy()
    kb = scores("kaggle", base.setting, base["mode"]).argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    drive_sub = os.path.join(RUN_DIR, f"submission_tta_airbench_{choice['mode']}_val{choice.val:.4f}_test{choice.test:.4f}.csv")
    sub.to_csv(drive_sub, index=False)
    os.makedirs(os.path.join(WORK_DIR, "tta"), exist_ok=True)
    local_sub = os.path.join(WORK_DIR, "tta", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"\nairbench TTA beat the current setting on VAL by > 0.2 pt -> saved {drive_sub}")
    print(f"Kaggle predictions that differ from the 0.915 submission: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)
else:
    print("\nairbench TTA did NOT beat the current setting on VAL by more than 0.2 pt -> keep the 0.915 submission; no new file written.")

computed 6 views x 3 sets in 587s
            setting mode    val   test
flip only (current)   sd 0.9852 0.9812
flip only (current)   ag 0.9828 0.9808
flip only (current) sdag 0.9836 0.9808
    airbench 6-view   sd 0.9848 0.9824
    airbench 6-view   ag 0.9828 0.9808
    airbench 6-view sdag 0.9828 0.9816

BASELINE (0.915 submission): flip only, mode 'sd': val 0.9852 | internal test 0.9812   (expected 0.9852 / 0.9812)
BEST AIRBENCH ON VAL: mode 'sd': val 0.9848 | internal test 0.9824 -> internal test change +0.12 pt (+3 images of 2,500)

airbench TTA did NOT beat the current setting on VAL by more than 0.2 pt -> keep the 0.915 submission; no new file written.
